ANALISI SUI PAESI E SUI FILM
-




In [9]:
import sqlalchemy
from sqlalchemy import create_engine
from sqlalchemy import inspect
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt
import geopandas as gpd
import geodatasets
import plotly.express as px

Collego le tabelle che contengono i dati da analizzare, da pgAdmin4 al Jupyter e prendo quelle di mio interesse (movies, countries, releases,genres, studios)

In [10]:
# Definiscire la connessione
username = "postgres"  #mettere il prorìprio username
password = "1234"    #attenzione mettere la propria password
host = "localhost"
port = "5432"
database = "Award_Explorer" #nome del database

# Creare l'engine di SQLAlchemy
engine = sqlalchemy.create_engine(f"postgresql://{username}:{password}@{host}:{port}/{database}")

# Creare un inspector per ottenere l'elenco delle tabelle
inspector = inspect(engine)
tables = inspector.get_table_names()

# Caricare tutte le tabelle sul DataFrame Pandas
dataframes = {table: pd.read_sql_table(table, con=engine) for table in tables}

# Mostrare i nomi delle tabelle caricate
#print(dataframes.keys())

# Accedere ai dati
movies_df = dataframes["movies"]
countries_df = dataframes["countries"]
releases_df = dataframes["releases"]
studios_df = dataframes["studios"]
genres_df = dataframes["genres"]

FACCIAMO I MERGE PER COMBINARE I DATI CHE CI SERVONO PER LE ANALISI

Merge per unire i dati di movies con countries usando id

In [11]:
# Unire i dati di 'movies' e 'countries' sul campo 'countries.id'
merged_data = pd.merge(movies_df, countries_df, how='left', left_on='id', right_on='id')

Merge per visualizzare quanti film ogni paese del mondo ha prodotto

In [12]:
# Contare il numero di film per paese
film_per_country = merged_data["country"].value_counts().reset_index()
film_per_country.columns = ["country", "num_films"]

#per visualizzare la tabella filtrata
film_per_country


,country,num_films
0,USA,174489
1,France,45725
2,UK,42914
3,Japan,41362
4,Germany,41325
...,...,...
242,Pitcairn,2
243,Heard Island and McDonald Islands,2
244,Cocos (Keeling) Islands,1
245,Norfolk Island,1


MAPPA SUL NUMERO DI FILM GIRATI DA OGNI PAESE DEL MONDO FINO AL 2024

In [14]:
# Creare la mappa coropletica con Plotly Express
fig = px.choropleth(
    film_per_country,
    locations="country",  # Nome della colonna con i paesi
    locationmode="country names",  # Interpreta i valori come nomi di paesi
    color="num_films",  # Numero di film girati
    title="Numero di Film Girati per Paese",
    color_continuous_scale="RdPu",  # Scala di colori (rosso-arancio)
    labels={'num_films': 'Numero di Film'},
)
# Aggiustare il layout della mappa: taglia figure e zoom
fig.update_layout(
    geo=dict(
        scope="world",             # mostra tutto il mondo
        projection_scale=1,      # zoom
    ),
    width=1000,                    # Larghezza della figura
    height=600                     # Altezza della figura
)
# Mostrare la mappa interattiva
fig.show()

commento sulla cartina

Merge e raggruppamenti per visualizzare i film prodotti da ogni paese dal 1990 al 2024

In [15]:
# Filtrare i dati per gli anni dal 2000 al 2024
movies_per_anno = merged_data[(merged_data['date'] >= 1990) & (merged_data['date'] <= 2024)]

# Conta il numero di film per paese e anno
film_per_country_per_year = movies_per_anno.groupby(['date', 'country']).size().reset_index(name='num_films')

#per visualizzare i film per anno di ogni paese
film_per_country_per_year

,date,country,num_films
0,1990.0,Afghanistan,2
1,1990.0,Albania,1
2,1990.0,Algeria,6
3,1990.0,Antarctica,1
4,1990.0,Argentina,19
...,...,...,...
5271,2024.0,Uruguay,24
5272,2024.0,Uzbekistan,5
5273,2024.0,Vietnam,18
5274,2024.0,Yugoslavia,1


MAPPA SUL NUMERO DI FILM PER PAESE DAL 1990 AL 2024

In [16]:
# Creare la mappa coropletica con Plotly Express (con animazione temporale)
fig = px.choropleth(
    film_per_country_per_year,
    locations="country",  # Nome della colonna con i paesi
    locationmode="country names",  # Interpreta i valori come nomi di paesi
    color="num_films",  # Numero di film girati
    title="Numero di Film Girati per Paese (1990-2024)",
    animation_frame="date",  # Cambia la mappa per anno
    color_continuous_scale=px.colors.sequential.Magma[::-1],  # Scala di colore (rosa/viola)
    labels={'num_films': 'Numero di Film'},
)

# Modificare il layout della mappa
fig.update_layout(
    geo=dict(
        scope="world",  # Mostra tutto il mondo
        projection_scale=1,  # zoom
    ),
    width=1000,  # Larghezza della figura
    height=600   # Altezza della figura
)

# Mostrare la mappa animata
fig.show()

commento sulla mappa